# 1. Library calling

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import datetime
import warnings
import numpy as np
from IPython.display import clear_output
# Ignore all warnings (not recommended in general)
warnings.filterwarnings("ignore")
import os
from tqdm import tqdm

# 2. Defining the Product Information and Location

In [16]:
SummaryFolder=r'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects'
summaryFile='Scraping_List.txt'
st=pd.read_csv(SummaryFolder+'\\'+summaryFile)
print(st)
#Define Product to extract
search_text = "Television"
print(search_text)
Source="Amazon"
OFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Outputs'
IFolder=fr'C:\Users\Vikram.Vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\{Source}\Inputs'
filename=Source+'_IN_ProductLinks_'+search_text+'.xlsx'
df1=pd.read_excel(IFolder+'\\'+filename)
imagepath=fr"C:\Users\Vikram.Vadhirajan\OneDrive - Trico\Vikram Data research\{search_text}\ProductImage"
if not os.path.exists(imagepath):
    os.makedirs(imagepath)
df1


                           Product Name
0                         Ignition Coil
1               Windshield Washer Pumps
2                 coupler trailer locks
3   Adjustable Trailer Hitch Ball Mount
4                        Vacuum Cleaner
..                                  ...
70               swing away hitch mount
71                      Garage Products
72                        Oxygen Sensor
73             Brand Specific O2 Sensor
74                                  FOB

[75 rows x 1 columns]
Television


,Links,Name,Sponsored?,ASINs
0,https://www.amazon.in/dp/B0FB35J4B6,LG 126 cm (50 inches) UA82 Series 4K Ultra HD ...,Sponsored,B0FB35J4B6
1,https://www.amazon.in/dp/B0FP2TB2C4,Wobble 138.7 cm (55 inches) K Series 4K UHD Sm...,Sponsored,B0FP2TB2C4
2,https://www.amazon.in/dp/B07MKFNHKG,VW 80 cm (32 inches) Frameless Series HD Ready...,NaN,B07MKFNHKG
3,https://www.amazon.in/dp/B0F84FBWQM,Samsung 80 cm (32 inches) HD Smart LED TV UA32...,NaN,B0F84FBWQM
4,https://www.amazon.in/dp/B07MNNH484,VW 80 cm (32 inches) Frameless Series HD Ready...,NaN,B07MNNH484
...,...,...,...,...
131,https://www.amazon.in/dp/B0DKNLRY8Q,acer 126 cm (50 inches) G Plus Series 4K Ultra...,NaN,B0DKNLRY8Q
132,https://www.amazon.in/dp/B0B9XT82V2,acer 139 cm (55 inches) W Series 4K Ultra HD Q...,NaN,B0B9XT82V2
133,https://www.amazon.in/dp/B0DZHMX6D3,TCL 80 cms (32 inches) V4C Series HD Ready Sma...,NaN,B0DZHMX6D3
134,https://www.amazon.in/dp/B0DW1C7X4C,VW 165 cm (65 inches) Pro Series 4K Ultra HD S...,NaN,B0DW1C7X4C


In [17]:
df1['Links']=df1['Links'].str.replace(".com",".in")
df1['Links'][1]

'https://www.amazon.in/dp/B0FP2TB2C4'

In [18]:
df1.loc[1,"Name"]

'Wobble 138.7 cm (55 inches) K Series 4K UHD Smart LED TV (Black) WB55UDAGU2875D25'

In [19]:
links=[]
for i in range(len(df1)):
    # if " " in df1.loc[i,"Name"]:
        links.append(df1['Links'][i])

links=set(links)
links=list(links)

In [20]:
length=len(links)
print(length)

136


# 3. Setting Webdriver and Website Specific Information

In [2]:
path= 'C://chromedriver.exe'
driver=webdriver.Chrome()
wait=WebDriverWait(driver, 5)
driver.get('https://www.amazon.in/')


In [8]:
location = driver.find_element(By.ID, 'nav-global-location-popover-link')

location.click()
sleep(3)
pin = driver.find_element(By.ID, 'GLUXZipUpdateInput')
pinnumber='562130'
pin.send_keys(pinnumber)

apply=driver.find_element(By.CSS_SELECTOR, 'span.a-button-inner[data-action="GLUXPostalUpdateAction"]')
apply.click()


# 4. Defining the Dataframe and Extracting the data into the Dataframe

In [13]:
cols =['Sl.No','Attributes','Short_Details',"Details"] #,'Review_Mentions'
df = pd.DataFrame(columns=cols)
df
count=0

In [21]:
driver.get(links[0])

In [22]:
driver.find_element(By.CSS_SELECTOR, '[class="a-price-whole"]').text

'1,339'

In [30]:
driver.find_element(By.CSS_SELECTOR, '[class="a-price a-text-price"]').text.replace("₹","")

'1,599'

In [31]:
driver.find_element(By.CSS_SELECTOR, '[class="a-size-small aok-offscreen"]').text

''

In [22]:
for i in tqdm(range(length)):
    driver.get(links[i])
    print(i)
    df.loc[count,'Sl.No']=int(i)
    sleep(3)
    df.loc[count,'Name']=driver.find_element(By.XPATH, "//span[@class='a-size-large product-title-word-break']").text
    df.loc[count,'Links']=links[i]
    try:
        pricetopay=float(driver.find_element(By.CSS_SELECTOR, '[class="a-price-whole"]').text.replace(",",""))
        df.loc[count,'Current Price']=pricetopay
    except:
        pass
    try:
        listprice=float(driver.find_element(By.CSS_SELECTOR, '[class="a-price a-text-price"]').text.replace("₹","").replace(",",""))
        df.loc[count,'List Price']=listprice
    except:
        try:
            df.loc[count,'List Price']=pricetopay
        except:
            pass
    try:
        df.loc[count,'Rating']=float(driver.find_element(By.ID, "averageCustomerReviews").text.split("\n")[0])
    except:
        df.loc[count,'Rating']=0               
    try:
        df.loc[count,'No of Ratings']=int(driver.find_element(By.ID, "acrCustomerReviewText").text.split(' ')[0].replace(",",""))
    except:
        df.loc[count,'No of Ratings']=0
    try:
        df.loc[count,"Product Description"]=(driver.find_element(By.XPATH, "(//div[@id='productDescription'])").text)
    except:
        df.loc[count,"Product Description"]=0

    Alist=[]
    try:
        table=driver.find_element(By.CSS_SELECTOR,'[id="productDetails_techSpec_section_1"]')
        Values=table.find_elements(By.TAG_NAME, "tr")
        for row in Values:
            # Extract data from columns (td/th elements)
            l=row.find_element(By.TAG_NAME, "th").text
            v=row.find_element(By.TAG_NAME, "td").text
            Alist.append(l+":"+v)
    except:
        pass
    try:
        table=driver.find_element(By.CSS_SELECTOR,'[id="productDetails_detailBullets_sections1"]')
        Values=table.find_elements(By.TAG_NAME, "tr")
        for row in Values:
            # Extract data from columns (td/th elements)
            l=row.find_element(By.TAG_NAME, "th").text
            v=row.find_element(By.TAG_NAME, "td").text
            Alist.append(l+":"+v)
    except:
        attrs=driver.find_element(By.CSS_SELECTOR,'[id="detailBulletsWrapper_feature_div"]').find_elements(By.CSS_SELECTOR,'[class="a-list-item"]')
        for att in attrs:
            Alist.append(att.text)
    df.at[count,'Attributes']=Alist
    variants=[]
    try:
        vs=len(driver.find_elements(By.CSS_SELECTOR,'[class="twisterSlotDiv "]'))
        for n in range(vs):

            variant=driver.find_elements(By.CSS_SELECTOR,'[class="twisterTextDiv text"]')[n].text
            cost=(driver.find_elements(By.CSS_SELECTOR,'[class="twisterSlotDiv "]')[n].text)
            ASIN=(driver.find_element(By.CSS_SELECTOR,f'[id="size_name_{n}"]').get_attribute('data-defaultasin'))
            v=variant+" :|: "+cost+" :|: "+ASIN
            variants.append(v)
        print(variants)
        sleep(1)
        df.at[count,'Variants']=variants
    except:
        df.at[count,'Variants']=""

    try:
        df.loc[count,'Sales_LastMonth']=int(driver.find_element(By.ID, "social-proofing-faceout-title-tk_bought").text.split("+")[0].replace('K','000'))
    except:
        df.loc[count,'Sales_LastMonth']=0
    try:
        df.loc[count,'Details']=driver.find_element(By.XPATH, "(//div[@class='a-section a-spacing-medium a-spacing-top-small'])").text
    except:
        ils=driver.find_elements(By.CSS_SELECTOR, '[class="a-unordered-list a-vertical a-spacing-small"]')
        ilist=[]
        for il in ils:
            ilist.append(il.text)
        df.at[count,'Details']=ilist
    D_s=[]
    try:
        ls=driver.find_elements(By.CSS_SELECTOR,'[class="a-span3"]')
        vs=driver.find_elements(By.CSS_SELECTOR,'[class="a-span9"]')
        for l,v in zip(ls,vs):
            D_s.append(l.text+":"+v.text)
    except:
        pass
    try:
        es=driver.find_elements(By.CLASS_NAME,"a-normal")
        os= es[2].find_elements(By.CLASS_NAME,"a-text-bold")
        vs=es[2].find_elements(By.CSS_SELECTOR,'[class="a-size-base handle-overflow"]')
        for o, v  in zip( os, vs):
            D_s.append(o.text+":"+v.text)
    except:
        pass
    try:
        Atts=driver.find_elements(By.CSS_SELECTOR,'[class="a-span6"]')
        for att in Atts:
            vals=att.text.replace("\n",':')
            D_s.append(vals)
    except:
        pass

    df.at[count,"Short_Details"]=D_s
    try:
        df.loc[count,'Product_Category']=driver.find_element(By.CSS_SELECTOR,'[class="a-unordered-list a-horizontal a-size-small"]').text.rsplit('\n')[-1]
    except:
        pass
    try:
        df.loc[count,'ReviewSummary']=driver.find_element(By.CSS_SELECTOR,'[id="product-summary"]').text
    except:
        pass
    try:
        df.loc[count,'FirstAvailableDate']=driver.find_element(By.XPATH, "//th[contains(text(), 'Date First Available')]").find_element(By.XPATH, "./following-sibling::td").text
    except:
        pass
    try:
        elements=driver.find_elements(By.CSS_SELECTOR, '[data-csa-c-action="infoPopOver"]')
        len(elements)
        slist=[]
        for element in elements:
            s=element.get_attribute('data-csa-c-item-id')
            slist.append(s)
        #print(slist)
        df.at[count,"Sentiment"]=slist
    except:
        df.loc[count,'Sentiment']=""
    try: 
        filename=links[i].split("/")[-1]+".jpg" # Extract filename from URL (modify if needed)
        t=driver.find_element(By.CSS_SELECTOR,'[id="landingImage"]')
        s=t.get_attribute('src')
        import requests

        # Download the image using requests
        response = requests.get(s, stream=True)

        if response.status_code == 200:            
            savepath=imagepath+"\\"+str(i)+"_"+filename
            df.loc[count,'ImageName']=str(i)+"_"+filename
        # Get filename from URL or generate a unique one
            with open(savepath, 'wb') as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
            print(f"Image downloaded: {filename}")
        else:
            print(f"Failed to download image: {s}")
    except:
        print(f"part number not available for {filename}")
    count=count+1
    balanceitem=length-i
    #print(balanceitem)
    #timeremaining(balanceitem,i)
    clear_output(wait=True)

100%|██████████| 136/136 [37:00<00:00, 16.33s/it]


In [23]:
print(df.shape)
df.tail()

(294, 17)


,Sl.No,Attributes,Short_Details,Details,Name,Links,Rating,No of Ratings,Product Description,Variants,Sales_LastMonth,Product_Category,FirstAvailableDate,Sentiment,ImageName,List Price,Current Price
289,131,"[Brand:coocaa, Manufacturer:Radiant Appliances...","[Screen Size:65 Inches, Brand:coocaa, Display ...",About this item\nResolution : 4K UHD TV (3840 ...,coocaa Frameless 164 cm (65 inch) Frameless QL...,https://www.amazon.in/dp/B0FVFSRDPV,4.0,0.0,0,[],0.0,Smart Televisions,9 October 2025,[],131_B0FVFSRDPV.jpg,11989.0,NaN
290,132,"[Brand:Philips, Manufacturer:Radiant Appliance...","[Screen Size:43 Inches, Brand:Philips, Display...",About this item\nResolution : UHD (3840x2160) ...,Philips 109 cm (43 inches) 8100 Series 4K Ultr...,https://www.amazon.in/dp/B0FDQY49H8,4.2,0.0,0,[],500.0,Smart Televisions,18 August 2025,[],132_B0FDQY49H8.jpg,11989.0,NaN
291,133,"[Brand:TCL, Manufacturer:TCL India, TTE ELECTR...","[Screen Size:75 Inches, Brand:TCL, Display Tec...",About this item\nResolution: 4K QLED (3840 x 2...,TCL 189 cm (75 inches) 4K Ultra HD Smart QLED ...,https://www.amazon.in/dp/B0F38JMQVR,4.0,0.0,0,[],100.0,Smart Televisions,27 May 2025,[],133_B0F38JMQVR.jpg,11989.0,NaN
292,134,"[Brand:XIAOMI, Manufacturer:Xiaomi, TTE Electr...","[Screen Size:43 Inches, Brand:XIAOMI, Display ...",About this item\nResolution : 4K Ultra HD (384...,Xiaomi 108 cm (43 inch) FX Ultra HD 4K Smart L...,https://www.amazon.in/dp/B0F3JP4TWW,4.1,0.0,0,[],1000.0,Smart Televisions,3 April 2025,[],134_B0F3JP4TWW.jpg,11989.0,NaN
293,135,"[Brand:TCL, Manufacturer:TCL India, TTE ELECTR...","[Screen Size:85 Inches, Brand:TCL, Display Tec...",About this item\nResolution: 4K QLED (3840 x 2...,TCL 215 cm (85 inches) 4K Ultra HD Smart QLED ...,https://www.amazon.in/dp/B0F38KZG56,4.0,0.0,0,[],0.0,Smart Televisions,27 May 2025,[],135_B0F38KZG56.jpg,11989.0,NaN


# 5. Post Processing Data and Exporting

In [24]:
df.columns

Index(['Sl.No', 'Attributes', 'Short_Details', 'Details', 'Name', 'Links',
       'Rating', 'No of Ratings', 'Product Description', 'Variants',
       'Sales_LastMonth', 'Product_Category', 'FirstAvailableDate',
       'Sentiment', 'ImageName', 'List Price', 'Current Price'],
      dtype='object')

In [45]:
df_at=df[["Name",'List Price','Attributes','Rating']][:157]
df_at

,Name,List Price,Attributes,Rating
0,"ASUS Gaming V16 (2025), 14th Gen,Intel Core 7 ...",NaN,"[Brand:ASUS, Manufacturer:ASUS, DIGITEK (CHONG...",5.0
1,acer Aspire AMD Ryzen 5-7430U Processor Laptop...,NaN,"[Brand:acer, Manufacturer:Acer, Tech Front Cho...",3.8
2,"HP Omnibook 5 OLED, Snapdragon X Processor (16...",NaN,"[Brand:HP, Manufacturer:HP, HP, HP India Sales...",4.1
3,"Dell 14, AMD Ryzen AI R7-350 Processor, 16GB L...",85965.0,"[Brand:Dell, Manufacturer:Dell India Pvt. Ltd....",3.7
4,HP OmniBook Ultra Flip (Previously Spectre) In...,NaN,"[Brand:HP, Manufacturer:HP, HP India Sales Pvt...",0.0
...,...,...,...,...
152,"acer Aspire, AMD Ryzen 7-7730U, 16GB RAM, 1TB ...",126220.0,"[Brand:acer, Manufacturer:Acer, Tech Front Cho...",3.4
153,Lenovo Legion Pro 5 2025 Intel Core Ultra 9 27...,85965.0,"[Brand:Lenovo, Manufacturer:Lenovo, One of the...",0.0
154,"ASUS TUF F16,14th Gen,Intel Core i7 14650HX,Ga...",126220.0,"[Brand:ASUS, Manufacturer:ASUS, INVENTEC (CHON...",4.4
155,Lenovo Thinkpad P16v Intel Core Ultra 9 185H 1...,126220.0,"[Brand:Lenovo, Manufacturer:LCFC (HeFei) Elect...",0.0


In [46]:
df_at=df[["Name",'Current Price','Attributes','Rating']][:157]
df_at=df_at.explode('Attributes')
df_at

,Name,Current Price,Attributes,Rating
0,"ASUS Gaming V16 (2025), 14th Gen,Intel Core 7 ...",NaN,Brand:ASUS,5.0
0,"ASUS Gaming V16 (2025), 14th Gen,Intel Core 7 ...",NaN,"Manufacturer:ASUS, DIGITEK (CHONGQING) Limited...",5.0
0,"ASUS Gaming V16 (2025), 14th Gen,Intel Core 7 ...",NaN,Series:ASUS V16,5.0
0,"ASUS Gaming V16 (2025), 14th Gen,Intel Core 7 ...",NaN,Colour:Matte Black,5.0
0,"ASUS Gaming V16 (2025), 14th Gen,Intel Core 7 ...",NaN,Form Factor:Clamshell,5.0
...,...,...,...,...
156,"HP 15, 13th Gen Intel Core i5-1334U, (16GB DDR...",NaN,Date First Available:14 April 2025,3.9
156,"HP 15, 13th Gen Intel Core i5-1334U, (16GB DDR...",NaN,Packer:HP India Sales Pvt. Ltd.,3.9
156,"HP 15, 13th Gen Intel Core i5-1334U, (16GB DDR...",NaN,Importer:HP India Sales Pvt. Ltd.,3.9
156,"HP 15, 13th Gen Intel Core i5-1334U, (16GB DDR...",NaN,Net Quantity:1 Count,3.9


In [61]:
df_at[['Attribute',"Value"]]=df_at['Attributes'].str.split(":", expand=True,n=1)


In [62]:
df_at_Pivoted=df_at.pivot_table(index=['Name','Current Price','Rating'], columns='Attribute', values='Value', aggfunc='first').reset_index()
df_at_Pivoted

Attribute,Name,Current Price,Rating,ASIN,ASIN,Are Batteries Included,Audio Details,Average Battery Life (in hours),Average Battery Standby Life (in hours),Batteries,...,Shape,Size,Speaker Description,Special Features,Specific Uses For Product,Standing screen display size,Total USB Ports,UPC,Voltage,Wireless Type
0,BKN® Mini Premium Metal Folding Portable Lapto...,284.0,4.2,B09CMHN77F,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,HP Omnibook (Previously Envy) X Flip OLED Inte...,106990.0,0.0,B07BS2GWTB,NaN,Yes,NaN,NaN,NaN,NaN,...,NaN,NaN,DTS:X® Ultra; Dual speakers; HP Audio Boost; P...,NaN,"Business, Student",35.6 Centimetres,4,191628505999 191628384372,NaN,"2.4 GHz Radio Frequency, 5 GHz Radio Frequency..."
2,"HP Smartchoice Omen AMD Ryzen 7 7840Hs, 8GB RT...",126220.0,4.0,B0CCVZP835,NaN,Yes,"Headphones, Speakers",8 Hours,7 Hours,1 Lithium Ion batteries required. (included),...,NaN,NaN,Audio by Bang & Olufsen; Dual speakers,NaN,NaN,40.9 Centimetres,NaN,NaN,NaN,802.11ax
3,"HP Spectre x360 AI Laptop, Intel Core Ultra 7 ...",172990.0,2.9,B0CRKN4D4S,NaN,Yes,Headphones,13 Hours,NaN,NaN,...,NaN,NaN,"Poly Studio, Quad speakers, HP Audio Boost, DT...",NaN,NaN,40.6 Centimetres,NaN,NaN,240 Volts,802.11ax
4,"Lenovo Legion 9 Intel Core i9-13980HX 16"" (40....",453914.0,0.0,B0DFCK3BZQ,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,16 Inches,NaN,NaN,NaN,"802.11ac, 802.11ax"
5,"Lenovo Legion 9 Intel Core i9-13980HX 16"" (40....",449500.0,2.7,B0DFCHZ522,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,16 Inches,NaN,NaN,NaN,NaN
6,"Lenovo Legion Pro 7 Intel Core i9-14900HX 16"" ...",167990.0,4.0,B0CXPXXHSH,NaN,Yes,Speakers,6 Hours,NaN,1 Lithium Polymer batteries required. (included),...,NaN,NaN,"Stereo speakers (super linear speaker), 2W x2,...",NaN,NaN,16 Inches,NaN,NaN,NaN,802.11ax
7,Lenovo {SmartChoice)Chromebook Intel Celeron N...,18999.0,3.9,B0F2TMZJ24,NaN,Yes,"Headphones, Speakers",10 Hours,16 Hours,1 D batteries required. (included),...,NaN,NaN,"Stereo speakers, 2W x2",NaN,NaN,11.6 Inches,NaN,NaN,NaN,"802.11ax, Bluetooth"
8,New Microsoft Surface Laptop (7th Edition) - W...,129990.0,5.0,B0D926BQL8,NaN,Yes,Speakers,13 Hours,20 Hours,1 Lithium Ion batteries required. (included),...,NaN,NaN,Dolby Atmos,NaN,NaN,13.8 Inches,NaN,NaN,15 Volts,802.11ax
9,Premium Vegan Leather Desk Mat 90X45cm 2.4mm T...,799.0,4.4,B0C2CC2FYR,NaN,NaN,NaN,NaN,NaN,NaN,...,Rectangular,90 x 45 cm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
# df = df[(df['Current Price'] > 0)]

In [48]:
df['ASIN']=df['Links'].str.split('dp/',expand=True)[1]
df['Product']=search_text
df=df.drop_duplicates(subset=['Links'])

In [49]:
df['Brand']=df["Short_Details"].apply(lambda x: str(x).replace('[','').replace(']','').replace('\'', '').replace(' ','')).str.split('Brand:',expand=True)[1].str.split(',',expand=True)[0]
df['Manufacturer']=df["Attributes"].apply(lambda x: str(x).replace('[','').replace(']','').replace('\'', '').replace(' ','')).str.split('Manufacturer:',expand=True)[1].str.split(',',expand=True)[0]
df['Source']="Amazon"
df[['Rating',"No of Ratings","Sales_LastMonth",'Manufacturer',"Brand","Source"]].head()

,Rating,No of Ratings,Sales_LastMonth,Manufacturer,Brand,Source
0,3.4,16.0,0.0,None,None,Amazon
1,3.5,3.0,0.0,DispoDrapes,None,Amazon
2,4.1,43.0,0.0,None,None,Amazon
3,3.2,767.0,50.0,Autofy,Autofy,Amazon
4,0.0,0.0,0.0,Optifit,Optifit,Amazon


In [50]:
cols=["Sl.No",
"Name",
"Product",
'Product_Category',
"Current Price",
"List Price",
"Rating",
"No of Ratings",
"Short_Details",
"Details",
"Product Description",
"Sales_LastMonth",
"Attributes",
"Brand",
"Part Number",
'Variants',
"Manufacturer",
"ReviewSummary",
"FirstAvailableDate",
"Sentiment",
"Material",
"ASIN",
"Links",
"Source",
"ImageName"
] #'Standards',
df=df[cols]

In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 153 entries, 0 to 152
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Sl.No               153 non-null    object 
 1   Name                153 non-null    object 
 2   Product             153 non-null    object 
 3   Product_Category    153 non-null    object 
 4   Current Price       153 non-null    float64
 5   List Price          153 non-null    float64
 6   Rating              153 non-null    float64
 7   No of Ratings       153 non-null    float64
 8   Short_Details       153 non-null    object 
 9   Details             153 non-null    object 
 10  Sales_LastMonth     153 non-null    float64
 11  Attributes          153 non-null    object 
 12  Brand               130 non-null    object 
 13  Manufacturer        138 non-null    object 
 14  FirstAvailableDate  119 non-null    object 
 15  ReviewSummary       43 non-null     object 
 16  Sentimen

In [52]:
print(df.shape)
df.head()

(153, 20)


,Sl.No,Name,Product,Product_Category,Current Price,List Price,Rating,No of Ratings,Short_Details,Details,Sales_LastMonth,Attributes,Brand,Manufacturer,FirstAvailableDate,ReviewSummary,Sentiment,ASIN,Links,Source
0,0,PUG IOCL Petrol Pump Uniform Fabric Cash bag,Motor Cycle Bags,Accessories,389.0,499.0,3.4,16.0,[],[],0.0,"[Date First Available : 14 April 2021, ASIN : ...",None,None,NaN,NaN,,B092JRJC9M,https://www.amazon.in/dp/B092JRJC9M,Amazon
1,1,Disposable sleeves cover Protective Sleeve wit...,Motor Cycle Bags,Arm Sleeves,289.0,370.0,3.5,3.0,[],Elastic at both ends (bicep and wrist) for arm...,0.0,"[Manufacturer:Dispo Drapes, Item part number:D...",None,DispoDrapes,NaN,NaN,[],B07CXJ4STJ,https://www.amazon.in/dp/B07CXJ4STJ,Amazon
2,2,XTS Gear Unisex Adult Xts Speedway Riding Jack...,Motor Cycle Bags,Jackets,4275.0,5999.0,4.1,43.0,[],"[IMPACT PROTECTION]: CE Level 2 shoulder, elbo...",0.0,"[Brand:XTS Gear, Model:XTS SPEEDWAY JACKET, Pa...",None,None,20 March 2022,"Customers say\nCustomers like the fit, quality...","[Appearance_POSITIVE, Fit_POSITIVE, Quality_PO...",B09W2LNXDT,https://www.amazon.in/dp/B09W2LNXDT,Amazon
3,3,Autofy Side Bag and Metal Clip for All Bikes (...,Motor Cycle Bags,Leather & Saddle Bags,468.0,499.0,3.2,767.0,"[Colour:Black, Size:One size, Brand:Autofy, Ma...",About this item\nVehicle Compatibility: All Bi...,50.0,"[Colour:Black, Size:One size, Brand:Autofy, Ma...",Autofy,Autofy,18 August 2016,Customers say\nCustomers like the value of the...,"[Value_POSITIVE, Quality_MIXED, Fit_MIXED, Loc...",B01KLP29PW,https://www.amazon.in/dp/B01KLP29PW,Amazon
4,4,"Optifit® Sport Hydration, 2. L Hydration Backp...",Motor Cycle Bags,Hydration Packs,1339.0,1949.0,0.0,0.0,"[Brand:Optifit, Colour:Black, Material:Down, S...",About this item\nLIGHTWEIGHT DURABILITY: Craft...,0.0,"[Tank Volume:2 Litres, Manufacturer:Optifit, c...",Optifit,Optifit,NaN,NaN,[],B0CXDDN4JY,https://www.amazon.in/dp/B0CXDDN4JY,Amazon


In [28]:
dfAtt=df[['ASIN','Attributes']]
dfAtt=dfAtt.explode('Attributes')
dfAtt[['Attributes', 'Value']] = dfAtt['Attributes'].str.split(':',1, expand=True)


KeyError: "['ASIN'] not in index"

In [54]:
summary_df = pd.pivot_table(dfAtt, values='Value', index='Attributes',aggfunc='count').reset_index()
summary_df =summary_df.sort_values(by='Value',ascending=False)
summary_df=summary_df.rename(columns ={'Value':'No Products contains this Attribute'})
summary_df


,Attributes,No Products contains this Attribute
57,Manufacturer,235
51,Item Weight,161
22,Best Sellers Rank,145
14,ASIN,138
28,Country of Origin,134
...,...,...
10,#68 in Laptop Backpacks,0
12,#89 in Laptop Backpacks,0
13,#91 in Hydration Packs,0
1,#104 in Laptop Backpacks,0


In [56]:
with pd.ExcelWriter(OFolder+'\\'+f'{Source}_IN_ProductDetails_'+search_text+'.xlsx') as writer:  # doctest: +SKIP
    df.to_excel(writer,index=False, sheet_name='Raw')
    summary_df.to_excel(writer,index=False, sheet_name='Attribute_Summary')
    dfAtt.to_excel(writer,index=False, sheet_name='Attribute_Values')

# 99. Archived Codes

In [1]:
#Downloading images from amazon.
from selenium import webdriver
from selenium.webdriver.common.by import By
import openpyxl
import requests
from io import BytesIO
from openpyxl import Workbook
from openpyxl.drawing.image import Image

#--------------------------------------------------------------------------------------------------------------------------
il=[]
for i in range(length):
    driver.get(links[i])
    imagelink=driver.find_element(By.ID,"landingImage").get_attribute('src')
    il.append(imagelink)
    response = requests.get(imagelink)
    img_data = BytesIO(response.content)
    ASIN=links[i].split('dp/')[1]
    img_filename = f"{ASIN}.jpg"
    with open(img_filename, "wb") as img_file:
        img_file.write(img_data.getvalue())
    # # Create a DataFrame
    # df = pd.DataFrame({'Product Image URL': [imagelink]})
    # df.loc[count,'ASIN']=ASIN
    # # Save the DataFrame to an Excel file
    # excel_filename = "product_data.xlsx"
    # df.to_excel(excel_filename, index=False,)
    # wb = Workbook()
    # ws = wb.active
    # img = Image(img_filename)
    # ws.add_image(img, f'C{count+2}')
    # wb.save(excel_filename)
    balanceitem=length-i
    timeremaining(balanceitem,i)

NameError: name 'length' is not defined